In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import sys
import matplotlib.patches as mpatches
import matplotlib as mpl
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap


df_interactions = pd.read_csv('/Volumes/lab-windingm/home/users/cochral/LRS/AttractionRig/analysis/social-isolation/n10/clustered-interactions_pt2/cropped_interactions.csv')

df_cluster = pd.read_csv('/Volumes/lab-windingm/home/users/cochral/LRS/AttractionRig/analysis/social-isolation/n10/clustered-interactions_pt2/F15_KMAX3_DMAX4_NMIN1500--/pca-data3-F15-mcmodels4-Kmax3-Dmax4-Nmin1500-05-2026.csv')

df =  pd.merge(
            df_interactions, 
            df_cluster[['interaction_id', 'Yhat.idt.pca']], 
            on='interaction_id', 
            how='inner'
        )         


### AVOIDANCE EXAMPLES -- CONDITION S ONLY

Isolated larvae that close on each other, come near WITHOUT ever touching, and then one of them
swerves sharply away. An interaction has to pass all of this:

- closest approach is between 1.5 and 3.5 mm -- near, but never in contact
- the two drawn paths also stay at least 1 mm apart at ANY time, not just frame by frame. Without
  this, one larva can pass through where the other was a moment earlier: nothing touches, but the
  traces cross on the page and the panel looks like a contact
- the gap shrinks by at least 3 mm on the way in and opens by at least 3 mm on the way out, moving
  the right way on at least 70% of the frames in each half
- one larva is heading at the other before (within 60 deg of the bearing to it), heading away after
  (beyond 110 deg), and its own heading swings by at least 60 deg across the encounter
- that swing happens ABRUPTLY: at least 35 deg of heading change in a single frame, somewhere in the
  four frames either side of closest approach

The heading swing rules out a larva ploughing straight on while the other drifts past -- the bearing
changes either way, but only a real turn changes the larva's own heading. The single-frame condition
is what makes it a sharp veer rather than a slow arc. A larva that is barely moving is skipped, since
a heading computed from a 0.1 mm step is noise.

Ranked by how sharp the turn is.


In [ ]:
## FIND THE AVOIDANCE EVENTS

NODES = ['head', 'body', 'tail']
FRAMES = np.arange(-10, 11)
PRE = (FRAMES >= -8) & (FRAMES <= -3)
POST = (FRAMES >= 3) & (FRAMES <= 8)
SHARP_WIN = (FRAMES >= -4) & (FRAMES <= 4)


def angle_between(u, v):
    c = (u * v).sum(-1) / (np.linalg.norm(u, axis=-1) * np.linalg.norm(v, axis=-1) + 1e-12)
    return np.degrees(np.arccos(np.clip(c, -1, 1)))


def find_avoidance():
    NO_TOUCH_MM = 1.5     # closest approach never below this -- no contact at all
    NEAR_MAX_MM = 3.5     # but close enough that it reads as a near miss
    PATH_GAP_MM = 1.0     # the two drawn paths stay this far apart at ANY time, not just same-frame
    APPROACH_MM = 3.0     # how much the gap has to shrink on the way in
    DEPART_MM = 3.0       # and open again on the way out
    MONO = 0.7            # fraction of frames going the right way in each half
    TURN_MIN = 60.0       # deg; total swing of the turner's own heading
    SHARP_MIN = 35.0      # deg in a SINGLE frame -- what makes the turn sharp
    MIN_STEP_MM = 0.20    # the larva has to be moving for a heading to mean anything
    FACE_MAX = 60.0       # deg; heading vs bearing to the other, before
    AWAY_MIN = 110.0      # deg; heading vs bearing to the other, after
    SMOOTH_WIN = 3

    cols = [f'mm_Track_{t} {a}_{n}' for t in (1, 2) for n in NODES for a in ('x', 'y')]
    d_s = df[df['condition'] == 'S']

    rows = []
    for iid, g in d_s.groupby('interaction_id', sort=False):
        g = g.set_index('Normalized Frame').reindex(FRAMES)
        if g[cols].isna().any().any():
            continue
        d = g['min_distance'].values

        if d.min() < NO_TOUCH_MM or d.min() > NEAR_MAX_MM:
            continue
        if d[0] - d[10] < APPROACH_MM or d[-1] - d[10] < DEPART_MM:
            continue
        if (np.diff(d[:11]) < 0).mean() < MONO or (np.diff(d[10:]) > 0).mean() < MONO:
            continue

        # the drawn traces must not cross or graze, even at different times
        h1 = g[['mm_Track_1 x_head', 'mm_Track_1 y_head']].values
        h2 = g[['mm_Track_2 x_head', 'mm_Track_2 y_head']].values
        path_gap = np.sqrt(((h1[:, None, :] - h2[None, :, :]) ** 2).sum(-1)).min()
        if path_gap < PATH_GAP_MM:
            continue

        xy = g[cols].values.reshape(len(FRAMES), 2, len(NODES), 2)
        body = pd.DataFrame(xy[:, :, 1, :].reshape(len(FRAMES), 4)) \
                 .rolling(SMOOTH_WIN, center=True, min_periods=1).mean() \
                 .values.reshape(len(FRAMES), 2, 2)

        best = None
        for k in (0, 1):
            vel = np.gradient(body[:, k, :], axis=0)
            speed = np.linalg.norm(vel, axis=1)
            bearing = body[:, 1 - k, :] - body[:, k, :]
            theta = angle_between(vel, bearing)
            turn = angle_between(vel[PRE].mean(0), vel[POST].mean(0))

            step = angle_between(vel[:-1], vel[1:])
            moving = (speed[:-1] >= MIN_STEP_MM) & (speed[1:] >= MIN_STEP_MM) & SHARP_WIN[:-1]
            sharp = step[moving].max() if moving.any() else 0.0

            t_pre, t_post = theta[PRE].mean(), theta[POST].mean()
            if (t_pre <= FACE_MAX and t_post >= AWAY_MIN
                    and turn >= TURN_MIN and sharp >= SHARP_MIN):
                if best is None or sharp > best[0]:
                    best = (sharp, k, turn, t_pre, t_post)
        if best is None:
            continue

        sharp, k, turn, t_pre, t_post = best
        rows.append(dict(interaction_id=iid, turner=k + 1, sharp_deg=sharp, turn_deg=turn,
                         face_before=t_pre, face_after=t_post, min_dist=d.min(),
                         path_gap=path_gap, approach_mm=d[0] - d[10], depart_mm=d[-1] - d[10],
                         interaction_type=g['interaction_type'].iloc[10],
                         cluster=g['Yhat.idt.pca'].iloc[10]))

    return pd.DataFrame(rows).sort_values('sharp_deg', ascending=False).reset_index(drop=True)


avoidance = find_avoidance()
print('avoidance events found:', len(avoidance))
lo, hi = avoidance['min_dist'].min(), avoidance['min_dist'].max()
print(f'closest approach: {lo:.2f} - {hi:.2f} mm   (nothing under 1 mm, nothing touching)')
avoid_sel = avoidance.head(20).reset_index(drop=True)
avoid_sel.round(2)


In [ ]:
## TRACE HELPERS (same treatment as the figure-1 example interactions)

from scipy.interpolate import splprep, splev

CORE_LW = 1.4
HALO_W = 8
N_HALO = 16
HALO_ALPHA = 0.045
HALO_TINT = 0.20   # how much white the soft band mixes in
CONTRAST = 3.0
RAMP_FLOOR = 0.22   # never start fully pale -- keeps the colour off the washed-out end
CONTACT_DIST = 1
SMOOTH_WIN = 3
BODY_MIX = 0.30
CONTACT_SIGMA = 1.5
SPLINE_PTS = 600

mpl.rcParams.update({'pdf.fonttype': 42, 'ps.fonttype': 42, 'svg.fonttype': 'none',
                     'font.family': 'sans-serif', 'font.sans-serif': ['Arial']})

# temporal ramp: pale at the start of the interaction, full colour at the end.
# built in HSV so hue is fixed and only saturation/brightness move -- blending toward
# white in RGB drains the chroma and the middle of the ramp comes out muddy.
import colorsys

TRACK_1_BASE = '#c25e38'   # a cleaner neighbour of aa5441; aa5441 itself reads brown next to the red
# TRACK_1_BASE = '#aa5441' # your original
# TRACK_1_BASE = '#b8593c' # between the two
# TRACK_1_BASE = '#cc6633' # brighter, starts to go orange
TRACK_2_BASE = '#e8443a'   # bright red
TINT_S = 0.22              # saturation at the pale end of the ramp
TINT_V = 1.00              # brightness at the pale end
RAMP_STOPS = 32


def hsv_ramp(base_hex):
    """light -> base with the hue held fixed, so the ramp never greys out"""
    r, g, b = [int(base_hex[i:i + 2], 16) / 255 for i in (1, 3, 5)]
    h, s, v = colorsys.rgb_to_hsv(r, g, b)
    return [colorsys.hsv_to_rgb(h, TINT_S + (s - TINT_S) * f, TINT_V + (v - TINT_V) * f)
            for f in np.linspace(0, 1, RAMP_STOPS)]


CMAPS = {1: LinearSegmentedColormap.from_list('t1', hsv_ramp(TRACK_1_BASE)),
         2: LinearSegmentedColormap.from_list('t2', hsv_ramp(TRACK_2_BASE))}


def ramp(t):
    t = np.clip(t, 0, 1)
    s = t ** CONTRAST / (t ** CONTRAST + (1 - t) ** CONTRAST + 1e-12)
    return RAMP_FLOOR + s * (1 - RAMP_FLOOR)


def lighten(c):
    o = c.copy()
    o[:, :3] = o[:, :3] + (1 - o[:, :3]) * HALO_TINT
    return o


def smooth_path(ex, tr):
    h = ex[[f'mm_Track_{tr} x_head', f'mm_Track_{tr} y_head']].values
    b = ex[[f'mm_Track_{tr} x_body', f'mm_Track_{tr} y_body']].values
    roll = lambda a: pd.DataFrame(a).rolling(SMOOTH_WIN, center=True, min_periods=1).mean().values
    sm = (1 - BODY_MIX) * roll(h) + BODY_MIX * roll(b)
    contact = np.where(ex['min_distance'].values < CONTACT_DIST)[0]
    idx = np.arange(len(ex))
    if len(contact):
        gap = np.abs(idx[:, None] - contact[None, :]).min(axis=1)
        w = np.exp(-(gap ** 2) / (2 * CONTACT_SIGMA ** 2))[:, None]
    else:
        w = np.zeros((len(ex), 1))
    return w * h + (1 - w) * sm


def spline(pts):
    keep = np.r_[True, (np.diff(pts, axis=0) ** 2).sum(1) > 1e-12]
    pts = pts[keep]
    tck, u = splprep(pts.T, s=0, k=min(3, len(pts) - 1))
    ue = np.linspace(0, 1, SPLINE_PTS)
    return np.column_stack(splev(ue, tck)), np.interp(ue, u, np.linspace(0, 1, len(pts)))


def trace(ax, pts, tr):
    xy, t = spline(pts)
    segs = np.stack([xy[:-1], xy[1:]], axis=1)
    cols = CMAPS[tr](ramp(t[:-1]))
    pale = lighten(cols)
    for w in np.linspace(HALO_W, CORE_LW * 1.5, N_HALO):
        ax.add_collection(LineCollection(segs, colors=pale, linewidths=w, alpha=HALO_ALPHA,
                                         capstyle='round', joinstyle='round', zorder=2))
    ax.add_collection(LineCollection(segs, colors=cols, linewidths=CORE_LW,
                                     capstyle='round', joinstyle='round', zorder=3))


def scale_bar(ax):
    MM = 1
    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    bx = x1 - 0.06 * (x1 - x0); by = y0 + 0.06 * (y1 - y0)
    ax.plot([bx - MM, bx], [by, by], color='0.25', lw=1.3, solid_capstyle='butt', zorder=6)
    ax.text(bx - MM / 2, by - 0.03 * (y1 - y0), f'{MM} mm', ha='center', va='top',
            fontsize=5.5, color='0.25')


In [ ]:
## THE PANEL -- 20 AVOIDANCE EVENTS, ONE COMMON SCALE

%config InlineBackend.figure_format = 'retina'


def draw_avoidance():
    NCOLS = 5
    PANEL_W_IN = 2.3
    PANEL_H_IN = 2.5
    PAD_MM = 1.5
    SAVE = None     # e.g. '/Users/cochral/repos/behavioural-analysis/plots/lrs_paper/FIGURE-2/avoidance-S.pdf'

    exs = {i: df[df['interaction_id'] == i].sort_values('Frame')
           for i in avoid_sel['interaction_id']}
    paths = {i: [smooth_path(e, tr) for tr in (1, 2)] for i, e in exs.items()}
    half = (max(max(np.ptp(np.vstack(p)[:, 0]), np.ptp(np.vstack(p)[:, 1]))
                for p in paths.values()) + 2 * PAD_MM) / 2

    nrows = int(np.ceil(len(avoid_sel) / NCOLS))
    fig, axes = plt.subplots(nrows, NCOLS, figsize=(PANEL_W_IN * NCOLS, PANEL_H_IN * nrows))
    axes = np.atleast_1d(axes).ravel()

    for ax, (_, row) in zip(axes, avoid_sel.iterrows()):
        iid = row.interaction_id
        for tr in (1, 2):
            trace(ax, paths[iid][tr - 1], tr)
        pts = np.vstack(paths[iid])
        cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
        ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half)
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f'{iid}\nturn {row.turn_deg:.0f}$\\degree$  sharpest {row.sharp_deg:.0f}$\\degree$/frame\nclosest {row.min_dist:.2f} mm',
                     fontsize=6.5, color='0.15', pad=3)
        scale_bar(ax)

    for ax in axes[len(avoid_sel):]:
        ax.axis('off')

    plt.tight_layout()
    if SAVE:
        fig.savefig(SAVE, bbox_inches='tight')
    plt.show()


draw_avoidance()


In [ ]:
## SCRATCH: THE NEXT 20 DOWN THE RANKING -- swap anything you do not like

START = 20


def show_more():
    global avoid_sel
    avoid_sel = avoidance.iloc[START:START + 20].reset_index(drop=True)
    draw_avoidance()


# show_more()
avoidance[['interaction_id', 'turn_deg', 'min_dist', 'approach_mm', 'depart_mm',
           'sharp_deg', 'path_gap', 'interaction_type', 'cluster']].head(40).round(2)


In [ ]:
## THESE FOUR ON THEIR OWN -- same traces as the panel above, just bigger

%config InlineBackend.figure_format = 'retina'


def draw_picked():
    IDS = ['S_2374', 'S_1562', 'S_464', 'S_2334']
    PANEL_W_IN = 3.4
    PANEL_H_IN = 3.8
    PAD_MM = 1.5
    SAVE = None     # e.g. '/Users/cochral/repos/behavioural-analysis/plots/lrs_paper/FIGURE-2/avoidance-picked.pdf'

    sel = avoidance.set_index('interaction_id').loc[IDS].reset_index()
    exs = {i: df[df['interaction_id'] == i].sort_values('Frame') for i in IDS}
    paths = {i: [smooth_path(e, tr) for tr in (1, 2)] for i, e in exs.items()}
    half = (max(max(np.ptp(np.vstack(p)[:, 0]), np.ptp(np.vstack(p)[:, 1]))
                for p in paths.values()) + 2 * PAD_MM) / 2

    fig, axes = plt.subplots(1, len(IDS), figsize=(PANEL_W_IN * len(IDS), PANEL_H_IN))
    axes = np.atleast_1d(axes).ravel()

    for ax, (_, row) in zip(axes, sel.iterrows()):
        iid = row.interaction_id
        for tr in (1, 2):
            trace(ax, paths[iid][tr - 1], tr)
        pts = np.vstack(paths[iid])
        cx, cy = pts[:, 0].mean(), pts[:, 1].mean()
        ax.set_xlim(cx - half, cx + half); ax.set_ylim(cy - half, cy + half)
        ax.set_aspect('equal'); ax.axis('off')
        ax.set_title(f'{iid}\nturn {row.turn_deg:.0f}$\\degree$   closest {row.min_dist:.2f} mm',
                     fontsize=8, color='0.15', pad=4)
        scale_bar(ax)

    plt.tight_layout()
    if SAVE:
        fig.savefig(SAVE, bbox_inches='tight')
    plt.show()


draw_picked()
